In [1]:
# ============================================================
# BUILD FACT_SESSION_RESULTS — LOCAL GOLD
# ============================================================

import os
from pyspark.sql import functions as F

In [2]:
try:
    SILVER_ROOT
except NameError:
    import nbformat
    from nbconvert import PythonExporter
    project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
    config_path = os.path.join(project_root, "/Users/manoharazalki/F1-Analytics/notebooks/00-common/01_environment_config.ipynb")
    with open(config_path, "r") as f:
        nb = nbformat.read(f, as_version=4)
    exporter = PythonExporter()
    source, _ = exporter.from_notebook_node(nb)
    exec(source)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/08 23:09:23 WARN Utils: Your hostname, Manohars-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.68.78.178 instead (on interface en0)
26/06/08 23:09:23 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/08 23:09:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/08 23:09:24 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/08 23:09:24 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/06/08 23:09:24 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/06/08 23:09:24 

✔ Spark Session Initialized
✔ Environment Config Loaded
PROJECT_ROOT: /Users/manoharazalki/F1-ANALYTICS


In [3]:
silver_results = f"{SILVER_ROOT}/results/results_silver.parquet"
silver_sprints = f"{SILVER_ROOT}/sprints/sprints_silver.parquet"
gold_folder = f"{GOLD_ROOT}/fact_session_results"
gold_file = f"{gold_folder}/fact_session_results.parquet"
os.makedirs(gold_folder, exist_ok=True)

In [4]:
results_df = (
    spark.read.parquet(silver_results)
        .withColumn("session_type", F.lit("RACE"))
        .withColumnRenamed("finish_position_text", "final_position_text")
)

sprints_df = (
    spark.read.parquet(silver_sprints)
        .withColumn("session_type", F.lit("SPRINT"))
)

results_sprints_df = results_df.unionByName(sprints_df)

In [5]:
fact_session_results_df = (
    results_sprints_df
        .withColumn("is_win", F.col("final_position") == 1)
        .withColumn("is_podium", F.col("final_position").between(1, 3))
        .withColumn("has_points", F.col("points") > 0)
)

In [6]:
(
    fact_session_results_df
        .write
        .mode("overwrite")
        .parquet(gold_file)
)

print(f"✔ fact_session_results written to: {gold_file}")
spark.read.parquet(gold_file).show(10, truncate=False)

✔ fact_session_results written to: /Users/manoharazalki/F1-ANALYTICS/data/gold/fact_session_results/fact_session_results.parquet
+------+-----+--------------+-----------+----------+------------------+-------------+--------------+----------+------+--------------+-------------------+------------+--------------------------+-------------------------------------------------------------+------------+------+---------+----------+
|season|round|constructor_id|driver_id  |race_date |race_name         |grid_position|completed_laps|car_number|points|final_position|final_position_text|status      |ingest_timestamp          |source_file                                                  |session_type|is_win|is_podium|has_points|
+------+-----+--------------+-----------+----------+------------------+-------------+--------------+----------+------+--------------+-------------------+------------+--------------------------+-------------------------------------------------------------+------------+------+--